# Text State Monitor (From S3)

Pulls `text_downloader.sqlite` from S3 every 60s and prints status progress.
Stop the cell when done.

In [ ]:
%pip install -q -U boto3 botocore
print('Installed boto3/botocore')


In [ ]:
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_REGION'] = userdata.get('AWS_REGION')
os.environ['S3_BUCKET'] = userdata.get('S3_BUCKET')

try:
    tok = userdata.get('AWS_SESSION_TOKEN')
    if tok:
        os.environ['AWS_SESSION_TOKEN'] = tok
except Exception:
    pass

required = ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_REGION', 'S3_BUCKET']
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise ValueError(f'Missing required secrets: {missing}')

print('Secrets loaded.')

In [ ]:
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

S3_BUCKET = os.environ['S3_BUCKET']
S3_KEY = 'clipfarm/state/text_downloader.sqlite'
LOCAL_DB = Path('/tmp/text_downloader_from_s3.sqlite')
REFRESH_SECONDS = 60
RECENT_ROWS = 12

s3 = boto3.client('s3', region_name=os.environ.get('AWS_REGION'))

def pull_db_from_s3():
    try:
        meta = s3.head_object(Bucket=S3_BUCKET, Key=S3_KEY)
        s3.download_file(S3_BUCKET, S3_KEY, str(LOCAL_DB))
        return True, int(meta.get('ContentLength', 0)), meta.get('LastModified')
    except ClientError as exc:
        return False, 0, str(exc)

def read_snapshot(db_path: Path):
    uri = f'file:{db_path}?mode=ro'
    conn = sqlite3.connect(uri, uri=True, timeout=30)
    conn.execute('PRAGMA busy_timeout = 5000')

    total = conn.execute('SELECT COUNT(*) FROM processed_items').fetchone()[0]
    by_status = conn.execute('''
        SELECT status, COUNT(*) AS c
        FROM processed_items
        GROUP BY status
        ORDER BY c DESC, status ASC
    ''').fetchall()
    recent = conn.execute('''
        SELECT processed_at, video_id, status
        FROM processed_items
        ORDER BY processed_at DESC
        LIMIT ?
    ''', (RECENT_ROWS,)).fetchall()
    conn.close()
    return total, by_status, recent

while True:
    clear_output(wait=True)
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    print(f'[monitor] {now}')
    print(f'[monitor] s3://{S3_BUCKET}/{S3_KEY}')

    ok, size, last = pull_db_from_s3()
    if not ok:
        print(f'[monitor] pull failed: {last}')
        time.sleep(REFRESH_SECONDS)
        continue

    print(f'[monitor] pulled bytes={size} last_modified={last}')

    try:
        total, by_status, recent = read_snapshot(LOCAL_DB)
        print(f'processed_total: {total}')
        print('\nstatus counts:')
        for status, count in by_status:
            pct = (count / total * 100.0) if total else 0.0
            print(f'  {status:35} {count:8d} ({pct:6.2f}%)')

        print('\nlatest rows:')
        for processed_at, video_id, status in recent:
            print(f'  {processed_at}  {video_id}  {status}')
    except Exception as exc:
        print(f'[monitor] sqlite read error: {exc}')

    time.sleep(REFRESH_SECONDS)
